# `lib.ipynb` — cell library for MOEX × Chronos-2 runs

**This notebook is not meant to be executed top-to-bottom in production.** It's the canonical source of reusable cells. When you build `run.ipynb` (or any stage notebook), you copy/import the relevant cells from here.

Sections:
1. Imports & GPU
2. Config loader (YAML)
3. Drive mount + cache paths
4. ISS soft prefetcher (rate-limited, retry, idempotent, multi-stage aware)
5. Cache-only loaders (no API fall-through during experiments)
6. Panel build + covariates + calendar features
7. Chronos input builder (long form, future_df)
8. Walk-forward driver
9. Metrics (DA + binomial + Wilson CI, corr, amplitude, coverage)
10. Baselines (B0/B1/B2/B3)
11. Plot helpers
12. Stage orchestrator (top-level `run_stage(cfg)` glue)

Each section is one or two cells. Copy what `run.ipynb` needs; do not edit logic here in-place — propagate fixes from `run.ipynb` back here when they stabilise.

## 1. Imports & GPU

In [ ]:
!pip install -q chronos-forecasting "pandas[pyarrow]" requests matplotlib numpy tqdm pyyaml scipy
# finetuning_wip (uncomment when running fine-tune stages):
# !pip install -q "autogluon.timeseries[chronos]"


In [ ]:
import os, math, json, time, warnings, hashlib, random
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional, Sequence, Any

import numpy as np
import pandas as pd
import requests
import yaml
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy import stats as sstats

warnings.filterwarnings("ignore", category=FutureWarning)

HAS_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if HAS_CUDA else "cpu"
if HAS_CUDA:
    cap = torch.cuda.get_device_capability(0)
    DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
    print(f"GPU: {torch.cuda.get_device_name(0)}  cap={cap}  dtype={DTYPE}")
else:
    DTYPE = torch.float32
    print("No CUDA — CPU only (slow).")


## 2. Config loader

YAML config with `.yaml` extension. The loader returns a plain `dict`; downstream code reads keys directly so config schema can evolve without breaking dataclass migrations. Required top-level keys are validated up front.

In [ ]:
REQUIRED_KEYS = [
    "stage_id", "interval", "tickers", "date_from", "date_till",
    "context_len", "horizon", "walk_forward", "covariates", "output_dir",
]

def load_config(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    missing = [k for k in REQUIRED_KEYS if k not in cfg]
    if missing:
        raise KeyError(f"config {path} missing required keys: {missing}")
    assert cfg["context_len"] <= 8192, "Chronos-2 max context is 8192"
    assert cfg["horizon"] <= 1024, "Chronos-2 max prediction_length is 1024"
    assert cfg["interval"] in (10, 60, 24), "interval must be 10, 60, or 24 (ISS has no 15m candle)"
    if cfg.get("data_source") == "algopack":
        assert cfg.get("algopack_processed_path"), \
            "data_source: algopack requires algopack_processed_path (path to data_pipeline's processed/<dataset>[_<interval>]/<group>.parquet)"
        # The configured path is written relative to a flat project layout
        # (Colab: MyDrive/moex-hack/{lib.ipynb, data_pipeline/, ...} --
        # data_pipeline/ sits directly next to lib.ipynb). A local checkout
        # of this repo has data_pipeline/ two levels up instead (this notebook lives in
        # forecasting/directional_experiment/). Rather than keep two
        # config variants, resolve at load time: if the configured path
        # doesn't exist as given but does exist one or two directories up, use that.
        # Leaves the flat-layout (Colab) case untouched -- it resolves on the
        # first try and never reaches the fallback.
        p = Path(cfg["algopack_processed_path"])
        if not p.exists():
            for up in (Path(".."), Path("../..")):
                if (up / p).exists():
                    cfg["algopack_processed_path"] = str(up / p)
                    break
    assert cfg.get("group_mode", "univariate") in ("univariate", "multivariate"), \
        "group_mode must be 'univariate' or 'multivariate' (Chronos-2 predict_df's cross_learning flag)"
    cfg.setdefault("indexes", ["IMOEX", "MOEXOG", "MOEXMM", "MOEXFN", "RGBI"])
    cfg.setdefault("futures_proxies", [])
    cfg.setdefault("quantiles", [0.1, 0.5, 0.9])
    cfg.setdefault("eval_horizons", [1, 2, 3, 5])
    cfg.setdefault("primary_horizons", [2, 3, 5])
    cfg.setdefault("baselines", ["zero", "last", "momentum5", "ar1"])
    cfg.setdefault("group_mode", "univariate")
    cfg.setdefault("min_ticker_coverage", 0.98)
    return cfg

def save_config_snapshot(cfg: dict, out_dir: str):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    with open(Path(out_dir) / "config.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

## 3. Drive mount + cache paths

Cache lives on Drive so it survives runtime resets. The same `cache_dir` is shared across stages — files are keyed by (secid, engine, interval, date_from, date_till) so a daily 2021–2026 SBER pull is reused by every stage that needs it.

In [ ]:
def mount_drive(mount_at: str = "/content/drive") -> bool:
    try:
        from google.colab import drive
        if not os.path.ismount(mount_at):
            drive.mount(mount_at, force_remount=False)
        return True
    except Exception as e:
        print(f"Not in Colab or mount failed ({e}); falling back to local cache.")
        return False

def resolve_cache_dir(cfg: dict) -> str:
    # Priority: explicit cfg.cache_dir → Drive default → local ./moex_cache
    if "cache_dir" in cfg and cfg["cache_dir"]:
        path = cfg["cache_dir"]
    elif mount_drive():
        path = "/content/drive/MyDrive/moex_cache"
    else:
        path = "./moex_cache"
    Path(path).mkdir(parents=True, exist_ok=True)
    print(f"cache_dir = {path}")
    return path


## 4. ISS soft prefetcher

Goals:
- **Soft on ISS**: ≥ 0.25 s between requests, retry on 429/5xx with exponential backoff, never more than 3 retries.
- **Idempotent**: skip files already on disk. Re-run safe.
- **Multi-stage aware**: takes a *list* of (engine, market, secid, interval, date_from, date_till) tuples and walks them all. The runner builds that list from the union of every stage's config.
- **No silent failures**: every miss is logged with the exact URL and HTTP status.

The cache key includes engine and market so identical SECIDs across engines don't collide.

In [ ]:
ISS_BASE = "https://iss.moex.com/iss"
RATE_LIMIT_SEC = 0.25
RETRY_BACKOFF = (1, 3, 8)  # seconds

def _cache_key(engine: str, market: str, secid: str, interval: int, dfrom: str, dtill: str) -> str:
    return f"{engine}_{market}_{secid}_{interval}_{dfrom}_{dtill}.parquet"

def _iss_candles(engine: str, market: str, secid: str, dfrom: str, dtill: str, interval: int) -> pd.DataFrame:
    """Fetch candles from ISS. Returns a DataFrame — possibly empty (with columns)
    when ISS genuinely has no data for the key. Raises RuntimeError on transport
    failure (exhausted retries / persistent 5xx) so callers never confuse a
    network problem with a true empty; only true empties get negative-cached."""
    url = f"{ISS_BASE}/engines/{engine}/markets/{market}/securities/{secid}/candles.json"
    rows, cols, start = [], None, 0
    while True:
        params = {"from": dfrom, "till": dtill, "interval": interval, "start": start}
        r = None
        for attempt, wait in enumerate([0] + list(RETRY_BACKOFF)):
            if wait: time.sleep(wait)
            try:
                resp = requests.get(url, params=params, timeout=30)
                if resp.status_code in (429, 500, 502, 503, 504):
                    continue
                resp.raise_for_status()
                r = resp
                break
            except requests.RequestException as e:
                if attempt == len(RETRY_BACKOFF):
                    raise RuntimeError(f"ISS fetch failed for {secid}: {e}") from e
        if r is None:
            raise RuntimeError(f"ISS fetch failed for {secid}: exhausted retries "
                               f"(last status {resp.status_code})")
        time.sleep(RATE_LIMIT_SEC)
        data = r.json().get("candles", {"columns": [], "data": []})
        cols = data["columns"]
        chunk = data["data"]
        if not chunk: break
        rows.extend(chunk)
        if len(chunk) < 500: break
        start += len(chunk)
    df = pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols or ["begin"])
    if not df.empty:
        df["begin"] = pd.to_datetime(df["begin"])
        df = df.sort_values("begin").reset_index(drop=True)
    return df

def prefetch_one(engine, market, secid, interval, dfrom, dtill, cache_dir, force=False) -> pd.DataFrame:
    """Idempotent single-key fetch with a *negative cache*.

    Prints exactly one status line per call so a run log shows, for every key,
    whether ISS was touched or the data came off disk, where the parquet lives,
    and how many rows it holds:
      CACHE     read from disk, has data (no ISS hit)
      CACHE-NEG read from disk, known-empty marker (no ISS hit)
      ISS       fetched over the network, has data → written to cache
      ISS-EMPTY fetched over the network, no data → negative-cache marker written
      FAIL      transport failure (not cached; retried next run)

    Genuine empties (dead / out-of-window FORTS contracts) are stored as a
    zero-row parquet so each such key hits ISS at most once. Transport failures
    raise inside `_iss_candles` and are left uncached."""
    fname = Path(cache_dir) / _cache_key(engine, market, secid, interval, dfrom, dtill)
    if fname.exists() and not force:
        df = pd.read_parquet(fname)
        tag = "CACHE-NEG" if df.empty else "CACHE    "
        print(f"  {tag} {fname}  rows={len(df)}")
        return df
    try:
        df = _iss_candles(engine, market, secid, dfrom, dtill, interval)
    except RuntimeError as e:
        print(f"  FAIL      {fname}  {e}")   # transient — not cached, retried next run
        return pd.DataFrame()
    df.to_parquet(fname)   # write even when empty → negative cache
    tag = "ISS-EMPTY" if df.empty else "ISS      "
    print(f"  {tag} {fname}  rows={len(df)} -> cached")
    return df

def build_prefetch_manifest(cfgs: list[dict]) -> list[tuple]:
    """Union the needs of every stage config into a deduped manifest.

    Shares are skipped for configs with data_source == "algopack" -- those
    prices come from data_pipeline's Parquet (see load_from_algopack), not the ISS
    cache, so queuing them here would just waste ISS requests. Indexes/futures
    still always come from ISS regardless of data_source (data_pipeline's index/
    futures output isn't wired into load_stage_inputs -- only shares are).
    """
    seen, out = set(), []
    for cfg in cfgs:
        interval = cfg["interval"]; dfrom = cfg["date_from"]; dtill = cfg["date_till"]
        if cfg.get("data_source") != "algopack":
            for t in cfg["tickers"]:
                key = ("stock", "shares", t, interval, dfrom, dtill)
                if key not in seen: seen.add(key); out.append(key)
        for idx in cfg.get("indexes", []):
            key = ("stock", "index", idx, interval, dfrom, dtill)
            if key not in seen: seen.add(key); out.append(key)
        for fut_root in cfg.get("futures_proxies", []):
            # FORTS root → expand to all candidate SECIDs; ISS serves FORTS at the
            # same intervals as equities (10m/60m/1d), so fetch at the target interval.
            source_interval = interval
            for secid in _enumerate_contracts(fut_root, dfrom, dtill):
                key = ("futures", "forts", secid, source_interval, dfrom, dtill)
                if key not in seen: seen.add(key); out.append(key)
    return out

def prefetch_all(manifest: list[tuple], cache_dir: str):
    """Walk the manifest; resumes mid-list because each item is independently cached."""
    for engine, market, secid, interval, dfrom, dtill in tqdm(manifest, desc="prefetch"):
        prefetch_one(engine, market, secid, interval, dfrom, dtill, cache_dir)

## 4b. FORTS chain resolver

FORTS contracts roll on a fixed cadence (Brent monthly; Si and Gold quarterly
H/M/U/Z). To build a continuous covariate series the resolver (a) enumerates
all candidate contracts whose lifetime could overlap the requested window,
(b) pulls per-contract candles via `prefetch_one` (reuses the soft prefetcher
+ cache), (c) picks the front-month contract per bar by daily volume, (d)
emits a stitched DataFrame carrying `contract_id` so the covariate builder
can NaN the return at every roll boundary.

ISS serves FORTS candles at the same intervals as equities (10m / 60m / 1d),
so each contract is fetched directly at the target interval — no cross-interval
fallback. (Note: ISS has no 15m candle for any market; intraday experiments use
10m or 60m bars.)

Resolved output is cached separately from per-contract candles so a re-run
skips both layers.

In [ ]:
FORTS_LETTER_MAP = {
    "BR": "FGHJKMNQUVXZ",  # Brent — monthly, all 12 letters
    "Si": "HMUZ",          # USD/RUB — quarterly H/M/U/Z
    "GD": "HMUZ",          # Gold   — quarterly H/M/U/Z
}

def _enumerate_contracts(root: str, dfrom: str, dtill: str) -> list[str]:
    """Candidate SECIDs whose lifetime could overlap [dfrom, dtill].
    Si trades ~2y before expiry → backward buffer 2y; +1y forward.
    Year-digit collisions are only across decades; 2019..2027 stays unique.
    """
    if root not in FORTS_LETTER_MAP:
        raise ValueError(f"unknown FORTS root: {root!r}; expected {list(FORTS_LETTER_MAP)}")
    letters = FORTS_LETTER_MAP[root]
    y_from = pd.to_datetime(dfrom).year - 2
    y_till = pd.to_datetime(dtill).year + 1
    return [f"{root}{L}{str(y % 10)}"
            for y in range(y_from, y_till + 1)
            for L in letters]

def _resolve_futures_chain(root: str, interval: int, dfrom: str, dtill: str,
                            cache_dir: str) -> pd.DataFrame:
    """Continuous FORTS series for `root` over [dfrom, dtill].
    Output schema: [begin, close, volume, contract_id] at the target interval
    (ISS serves FORTS at 10m/60m/1d, same as equities). Covariate builder
    regrids close onto the target interval, carries contract_id along, and NaNs
    the return at every contract switch.
    Cached at: {root}_forts_resolved_{interval}_{dfrom}_{dtill}.parquet.
    """
    cache_fname = Path(cache_dir) / f"{root}_forts_resolved_{interval}_{dfrom}_{dtill}.parquet"
    if cache_fname.exists():
        return pd.read_parquet(cache_fname)

    source_interval = interval
    parts = []
    for secid in _enumerate_contracts(root, dfrom, dtill):
        df = prefetch_one("futures", "forts", secid, source_interval, dfrom, dtill, cache_dir)
        if df.empty: continue
        parts.append(df[["begin", "close", "volume"]].assign(contract_id=secid))
    if not parts:
        raise RuntimeError(f"_resolve_futures_chain: no contracts found for "
                           f"root={root!r} interval={source_interval} {dfrom}..{dtill}")

    all_rows = pd.concat(parts, ignore_index=True)
    # Front-month per bar: highest volume wins; deterministic tiebreak via stable sort.
    all_rows = all_rows.sort_values(["begin", "volume"], ascending=[True, False], kind="stable")
    resolved = (all_rows.drop_duplicates(subset=["begin"], keep="first")
                        .sort_values("begin").reset_index(drop=True))
    resolved["begin"] = pd.to_datetime(resolved["begin"])

    # Validation: no intra-contract |log-return| > 0.20.
    close = resolved["close"].astype(float)
    ret_check = np.log(close / close.shift(1))
    boundary = resolved["contract_id"] != resolved["contract_id"].shift(1)
    intra = ret_check[~boundary]
    spikes = intra[intra.abs() > 0.20]
    if len(spikes):
        print(f"  WARN {root}: {len(spikes)} intra-contract bars with |log_return|>0.20")
        print(resolved.loc[spikes.index].head(5)[["begin", "contract_id", "close"]].to_string())

    resolved.to_parquet(cache_fname)
    return resolved

## 5. Cache-only loaders

During experiment runs we never hit the ISS API. If a file is missing it's a config bug — we fail loudly with the exact key. Production / live forecasting flips `allow_api_fallback=True`.

`load_stage_inputs` also supports `cfg["data_source"] == "algopack"`: reads `data_pipeline`'s processed Parquet output via `load_from_algopack` instead of the ISS cache. Same output shape either way — everything downstream is unchanged.

In [ ]:
class CacheMiss(KeyError):
    pass

def load_cached(engine, market, secid, interval, dfrom, dtill, cache_dir, allow_api_fallback=False) -> pd.DataFrame:
    fname = Path(cache_dir) / _cache_key(engine, market, secid, interval, dfrom, dtill)
    if fname.exists():
        return pd.read_parquet(fname)
    if allow_api_fallback:
        print(f"  cache miss -> API: {fname.name}")
        return prefetch_one(engine, market, secid, interval, dfrom, dtill, cache_dir)
    raise CacheMiss(f"missing cache file: {fname}")

def load_from_algopack(processed_path, tickers, interval_label="1d") -> dict[str, pd.DataFrame]:
    """Adapter: reads a `data_pipeline` pipeline output Parquet (long format:
    `ticker, timestamp, open, high, low, close, volume, value`, tz-aware MSK)
    and reshapes it into the `{ticker: df}` dict of ISS-candle-shaped
    DataFrames (`begin, open, high, low, close, volume, value`) that
    `load_stage_inputs`/`build_price_panel`/`to_regular_series` already expect.

    `processed_path` is e.g. `<data_pipeline_output_root>/processed/candles_1d/shares.parquet`
    (see `data_pipeline/docs/usage.md` for the full output layout). No downstream code
    changes needed -- this function's only job is shape translation.
    """
    long_df = pd.read_parquet(processed_path)
    long_df = long_df[long_df["ticker"].isin(tickers)]
    out = {}
    for tic, g in long_df.groupby("ticker"):
        g = g.rename(columns={"timestamp": "begin"}).sort_values("begin").reset_index(drop=True)
        # to_regular_series/_session_grid expect naive-or-tz-consistent datetimes on `begin`;
        # data_pipeline outputs tz-aware Europe/Moscow, so drop tz to match the ISS cache's
        # naive-MSK convention used throughout the rest of this pipeline.
        if isinstance(g["begin"].dtype, pd.DatetimeTZDtype):
            g["begin"] = g["begin"].dt.tz_localize(None)
        out[tic] = g
    missing = set(tickers) - set(out)
    if missing:
        print(f"  load_from_algopack: no rows for {sorted(missing)} in {processed_path}")
    return out

def load_stage_inputs(cfg: dict, cache_dir: str) -> tuple[dict, dict, dict]:
    interval, dfrom, dtill = cfg["interval"], cfg["date_from"], cfg["date_till"]
    if cfg.get("data_source") == "algopack":
        # cfg["algopack_processed_path"] points at data_pipeline's processed/<dataset>[_<interval>]/<group>.parquet
        prices = load_from_algopack(cfg["algopack_processed_path"], cfg["tickers"])
    else:
        prices = {t: load_cached("stock","shares",t,interval,dfrom,dtill,cache_dir) for t in cfg["tickers"]}
    indexes = {}
    for idx in cfg.get("indexes", []):
        try:
            df = load_cached("stock","index",idx,interval,dfrom,dtill,cache_dir)
            if not df.empty: indexes[idx] = df
        except CacheMiss as e:
            print(f"  skipping index {idx}: {e}")
    futures = {}
    for fut_root in cfg.get("futures_proxies", []):
        # FORTS root -> continuous chain (see section 4b resolver).
        try:
            df = _resolve_futures_chain(fut_root, interval, dfrom, dtill, cache_dir)
            if not df.empty: futures[fut_root] = df
        except Exception as e:
            print(f"  skipping future {fut_root}: {e}")
    return prices, indexes, futures


## 6. Panel build + covariates + calendar features

Regularise per series, then inner-join across tickers so the panel index is identical for all ids (group attention requirement). Covariates are broadcast across all ids; calendar features only.

`build_price_panel` guards the inner join with `min_ticker_coverage` (default 0.98): a ticker whose raw (pre-ffill) bar coverage of the union calendar falls below threshold is dropped and logged, rather than silently shrinking every other ticker's usable date range. Matters at 50-100 ticker scale.

In [ ]:
def _session_grid(start, end, interval_min):
    days = pd.bdate_range(start.normalize(), end.normalize())
    bars_per_day = (8*60 + 50) // interval_min  # 10:00..18:50 MSK
    out = [pd.date_range(d + pd.Timedelta(hours=10), periods=bars_per_day, freq=f"{interval_min}min") for d in days]
    return pd.DatetimeIndex(np.concatenate(out)) if out else pd.DatetimeIndex([])

def to_regular_series(df, value_col, name, interval):
    if df.empty: return pd.Series(name=name, dtype=float)
    s = df.set_index("begin")[value_col].astype(float).sort_index()
    if interval == 24:
        s = s.asfreq("B")
    else:
        s = s.reindex(_session_grid(s.index.min(), s.index.max(), interval))
    return s.ffill().rename(name)

def _raw_bar_coverage(df, union_index, interval):
    """Fraction of `union_index` bars for which `df` has a genuine (non-ffilled)
    observation. Computed on the RAW `begin` timestamps, before to_regular_series's
    ffill hides gaps -- ffill makes every series look 100% "complete" on its own
    asfreq'd grid, so coverage must be measured pre-ffill to mean anything.
    """
    if df.empty:
        return 0.0
    raw_idx = pd.DatetimeIndex(pd.to_datetime(df["begin"]).unique())
    if interval != 24:
        # session-grid bars only count if they land exactly on a real 10:00-18:50 slot
        raw_idx = raw_idx.intersection(_session_grid(raw_idx.min(), raw_idx.max(), interval))
    hits = union_index.isin(raw_idx)
    return float(hits.sum()) / len(union_index) if len(union_index) else 0.0

def build_price_panel(prices, interval, value_col="close", min_ticker_coverage=0.98):
    """Regularise each ticker's series, then inner-join across all tickers.

    `dropna(how="any")` is an inner join on dates: one illiquid/gappy ticker
    silently shrinks the usable date range for *every* other ticker, and
    to_regular_series's ffill makes a gappy series look complete on its own
    grid (so coverage can't be measured post-ffill). Before the join, drop any
    ticker whose RAW (pre-ffill) bar coverage of the union calendar is below
    `min_ticker_coverage`, and log what was dropped and why -- so the failure
    mode is visible, not silent. `None` disables the guard (old behaviour).
    `value_col` selects which column of each ticker's df to use as price
    (e.g. "close_adj" for dividend/split-adjusted data_pipeline output); default
    "close" keeps existing configs behaving exactly as before.
    """
    regular = {t: to_regular_series(df, value_col, t, interval) for t, df in prices.items()}
    non_empty = {t: s for t, s in regular.items() if not s.empty}
    if min_ticker_coverage is not None and non_empty:
        union_index = non_empty[next(iter(non_empty))].index
        for s in non_empty.values():
            union_index = union_index.union(s.index)
        kept = {}
        for t, s in non_empty.items():
            coverage = _raw_bar_coverage(prices[t], union_index, interval)
            if coverage < min_ticker_coverage:
                print(f"  build_price_panel: dropping {t} (raw coverage={coverage:.3f} < {min_ticker_coverage})")
            else:
                kept[t] = s
        regular = kept
    else:
        regular = non_empty
    panel = pd.concat(regular.values(), axis=1).dropna(how="any") if regular else pd.DataFrame()
    if regular:
        union_days = len(pd.concat(regular.values(), axis=1).index)
        joined_days = len(panel)
        if union_days:
            shrink = 1 - joined_days / union_days
            tag = "WARN" if shrink > 0.10 else "info"
            print(f"  build_price_panel: {tag} inner-join kept {joined_days}/{union_days} "
                  f"days ({shrink:.1%} lost to gaps across {len(regular)} tickers)")
    return panel

def log_returns(panel):
    return np.log(panel / panel.shift(1)).dropna(how="any")

def calendar_features(index, interval):
    df = pd.DataFrame({
        "hour":  index.hour.astype(np.float32),
        "dow":   index.dayofweek.astype(np.float32),
        "dom":   index.day.astype(np.float32),
        "month": index.month.astype(np.float32),
    }, index=index)
    if interval != 24:
        df["session_open"] = ((index.hour >= 10) & (index.hour < 19)).astype(np.float32)
    return df

def build_covariate_panel(prices, indexes, futures, interval, ret_index):
    parts = []
    for name, df in indexes.items():
        s = to_regular_series(df,"close",f"{name}_close",interval)
        parts.append(np.log(s/s.shift(1)).rename(f"{name}_ret"))
    for name, df in futures.items():
        # Resolved chain: regrid close (ffill), carry contract_id, compute return
        # on the target interval, NaN at every roll boundary.
        s_close = to_regular_series(df, "close", f"{name}_close", interval)
        cid_src = df.set_index("begin")["contract_id"]
        s_cid = cid_src.reindex(s_close.index, method="ffill")
        ret = np.log(s_close / s_close.shift(1))
        boundary = (s_cid != s_cid.shift(1)).fillna(False)
        ret = ret.mask(boundary)
        parts.append(ret.rename(f"{name}_ret"))
    for tic, df in prices.items():
        v = to_regular_series(df,"volume",f"{tic}_vol",interval)
        parts.append(np.log1p(v).diff().rename(f"{tic}_dlogvol"))
    cov = pd.concat(parts, axis=1).reindex(ret_index).ffill().dropna(how="any")
    return cov

def assemble_panels(cfg, prices, indexes, futures):
    price = build_price_panel(prices, cfg["interval"], value_col=cfg.get("price_col", "close"), min_ticker_coverage=cfg.get("min_ticker_coverage", 0.98))
    ret = log_returns(price)
    cov_mode = cfg["covariates"]  # "full", "calendar_only", "market_only", "none"
    if cov_mode == "none":
        cov = pd.DataFrame(index=ret.index)
    else:
        # Only pass the tickers that survived build_price_panel's coverage guard --
        # build_covariate_panel adds one {ticker}_dlogvol column per entry in `prices`
        # and then does dropna(how="any") across ALL of them. A ticker the guard just
        # dropped for being too gappy (e.g. a 115-bar recent listing) still has a
        # ragged raw volume series; including it here silently collapses `cov` to
        # almost nothing (observed: 1301 rows -> 113 rows on the real Phase B pull),
        # which then collapses ret.index.intersection(cov.index) and produces "0
        # windows" downstream with no error -- caught via a real Colab run 2026-09-17.
        surviving_prices = {t: prices[t] for t in price.columns}
        cov = build_covariate_panel(surviving_prices, indexes, futures, cfg["interval"], ret.index)
        if cov_mode == "calendar_only":
            cov = cov.iloc[:, 0:0]  # empty market cov; calendar handled separately
    # realign
    common = ret.index.intersection(cov.index) if not cov.empty else ret.index
    return price.loc[common], ret.loc[common], cov.loc[common]


## 7. Chronos input builder

In [ ]:
def build_chronos_inputs(ret_panel, cov_panel, tickers, context_len, horizon, t_anchor,
                          interval, covariate_mode="full"):
    """
    t_anchor = first index of the forecast horizon (so context ends at t_anchor-1).
    Returns (context_df, future_df, fut_idx).
    """
    pos = ret_panel.index.get_loc(t_anchor)
    ctx_idx = ret_panel.index[pos-context_len:pos]
    fut_idx = ret_panel.index[pos:pos+horizon]

    use_market_cov = covariate_mode in ("full", "market_only") and not cov_panel.empty
    use_calendar = covariate_mode in ("full", "calendar_only")
    cal_ctx = calendar_features(ctx_idx, interval) if use_calendar else None
    cal_fut = calendar_features(fut_idx, interval) if use_calendar else None
    cov_ctx = cov_panel.loc[ctx_idx] if use_market_cov else None

    ctx_rows, fut_rows = [], []
    for tic in tickers:
        block = pd.DataFrame({"id": tic, "timestamp": ctx_idx, "target": ret_panel[tic].loc[ctx_idx].values})
        if use_market_cov:
            for c in cov_ctx.columns: block[c] = cov_ctx[c].values
        if use_calendar:
            for c in cal_ctx.columns: block[c] = cal_ctx[c].values
        ctx_rows.append(block)
        fb = pd.DataFrame({"id": tic, "timestamp": fut_idx})
        if use_calendar:
            for c in cal_fut.columns: fb[c] = cal_fut[c].values
        fut_rows.append(fb)
    return pd.concat(ctx_rows, ignore_index=True), pd.concat(fut_rows, ignore_index=True), fut_idx


## 8. Walk-forward driver

Loops over forecast windows, calls Chronos once per window (batched across all tickers
in one `predict_df` call), persists predictions to Parquet. Defensive on indexing —
`t_anchor` must be far enough from both edges of the panel.

**`group_mode` (Phase B gate, `cfg["group_mode"]`, default `"univariate"`):** controls
`predict_df`'s `cross_learning` flag. `"univariate"` (`cross_learning=False`) forecasts
every ticker independently even though they share one batched call — batching into one
`id_column`-keyed DataFrame is *not* the same as joint attention (verified against
chronos-forecasting's source: `cross_learning` controls whether each task's `group_id`
is zeroed so all tasks in the batch attend to each other). `"multivariate"`
(`cross_learning=True`) is the actual gate condition. Prior stages (0–2) never set this
flag, so they were already running in independent/univariate mode — Stage 2b's negative
result is a univariate baseline, not a multivariate one.

In [ ]:
def walk_forward_anchors(ret_index, context_len, horizon, shift, max_windows=None):
    starts = []
    pos = context_len
    end_pos = len(ret_index) - horizon
    while pos < end_pos:
        starts.append(ret_index[pos])
        pos += shift
    if max_windows and len(starts) > max_windows:
        # keep evenly spaced subsample if too many
        idx = np.linspace(0, len(starts)-1, max_windows).astype(int)
        starts = [starts[i] for i in idx]
    return starts

def run_walk_forward(pipeline, cfg, ret_panel, cov_panel, out_dir):
    preds_dir = Path(out_dir) / "preds"; preds_dir.mkdir(parents=True, exist_ok=True)
    wf = cfg["walk_forward"]
    anchors = walk_forward_anchors(
        ret_panel.index, cfg["context_len"], cfg["horizon"],
        shift=wf["shift"], max_windows=wf.get("max_windows"),
    )
    print(f"walk-forward: {len(anchors)} windows")
    all_preds = []
    for i, t_anchor in enumerate(tqdm(anchors, desc="windows")):
        ctx, fut, fut_idx = build_chronos_inputs(
            ret_panel, cov_panel, cfg["tickers"], cfg["context_len"], cfg["horizon"],
            t_anchor, cfg["interval"], cfg["covariates"],
        )
        try:
            # validate_inputs=False: the MOEX session grid (10:00..17:00 on
            # business days) is *not* a single regular frequency — it skips
            # overnight/weekend gaps. Chronos infers one freq and otherwise
            # rejects every horizon that crosses a day boundary ("future_df
            # timestamps do not match the expected prediction timestamps"),
            # which silently dropped ~half the windows (all afternoon anchors).
            # With the check off, future covariates are consumed positionally
            # (predict_df sorts each id by timestamp, then aligns by row order),
            # which matches the order we built `fut` / `fut_idx` in.
            #
            # group_mode / cross_learning: this is the actual multivariate-vs-
            # univariate switch (Phase B gate). cross_learning=False (default,
            # group_mode="univariate") forecasts every id in the batch
            # independently even though they're all in one predict_df call --
            # grouping into one `id_column`-keyed DataFrame does NOT by itself
            # mean joint attention (confirmed against chronos-forecasting's
            # source: cross_learning zeroes each task's group_id so they all
            # attend to each other; false leaves each task's own group_id
            # alone, i.e. independent). cross_learning=True (group_mode=
            # "multivariate") is the one line that changes between the two
            # Phase B arms -- covariate attachment above is identical either way.
            pred = pipeline.predict_df(
                ctx, future_df=fut, prediction_length=cfg["horizon"],
                quantile_levels=list(cfg["quantiles"]),
                id_column="id", timestamp_column="timestamp", target="target",
                validate_inputs=False,
                # freq=: predict_df's own make_future_df call runs UNCONDITIONALLY
                # (chronos/chronos2/pipeline.py, independent of validate_inputs) to infer a
                # pandas frequency from ctx's timestamps -- and MOEX's intraday session grid
                # (e.g. 10:00-18:50 at interval=10, gaps overnight/weekends) has no standard
                # pandas freq alias, so pd.infer_freq returns None and predict_df raises
                # "Could not infer frequency for series ..." on essentially every window (hit
                # for real on the 10-minute follow-on 2026-09-17 -- daily bars never hit this
                # because pandas recognizes "B" despite weekly gaps). The comment below about
                # relabelling positionally already means these freq-derived timestamps are
                # thrown away and overwritten from the real fut_idx immediately after this
                # call -- so any syntactically valid offset string is fine here, it only needs
                # to not crash to_offset(); it is never used for the actual forecast values
                # (those come from predict_quantiles/prepared, not from future/freq).
                freq=f"{cfg['interval']}min" if cfg["interval"] != 24 else "B",
                cross_learning=(cfg.get("group_mode", "univariate") == "multivariate"),
            )
        except Exception as e:
            print(f"  window {i} ({t_anchor}) failed: {e}")
            continue
        # With validation off, predict_df labels the output with its own naive
        # freq continuation (e.g. ...17:00, 18:00, 19:00) instead of the real
        # next session bars. Relabel positionally to the true `fut_idx` so the
        # metrics/plots look up the correct realised bars in `ret_panel`.
        pred = pred.sort_values(["id", "timestamp"]).reset_index(drop=True)
        h0 = pred.groupby("id", sort=False).cumcount().to_numpy()
        pred["timestamp"] = pd.DatetimeIndex(fut_idx)[h0]
        pred["window"] = i
        pred["t_anchor"] = t_anchor
        all_preds.append(pred)
        if (i+1) % 25 == 0:  # checkpoint flush
            pd.concat(all_preds, ignore_index=True).to_parquet(preds_dir / "preds_partial.parquet")
    full = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
    full.to_parquet(preds_dir / "preds.parquet")
    return full

## 9. Metrics

Per (ticker, horizon-step) we compute DA + binomial p + Wilson CI, Pearson r, Spearman ρ, |pred|/|true| median, q-coverage. Then bootstrap aggregates across tickers.

`trend2_metrics` (next cell) is a separate, window-level view: it isolates windows where Chronos predicts a 2-bar trend (median return of bar 1 and bar 2 share a sign, +,+ or -,-) and reports how often the realised returns agree — strict `both` (both real bars match the sign, chance 0.25) and cumulative `cum` (sign of r1+r2 matches, chance 0.50).

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n))/denom
    half = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/denom
    return (center-half, center+half)

def per_cell_metrics(pred_df, ret_panel, tickers, horizons, quantiles):
    if pred_df.empty or "id" not in pred_df.columns:
        return pd.DataFrame()
    median_q = str(quantiles[len(quantiles)//2])
    lo_q, hi_q = str(quantiles[0]), str(quantiles[-1])
    rows = []
    for tic in tickers:
        sub = pred_df[pred_df["id"] == tic].copy()
        if sub.empty: continue
        # per window: which horizon step is this row? step = 1..H within window
        sub = sub.sort_values(["window", "timestamp"])
        sub["h_step"] = sub.groupby("window").cumcount() + 1
        for h in horizons:
            slab = sub[sub["h_step"] == h]
            if slab.empty: continue
            y_pred = slab[median_q].values
            ts = slab["timestamp"].values
            try:
                y_true = ret_panel[tic].loc[pd.DatetimeIndex(ts)].values
            except KeyError:
                continue
            mask = ~(np.isnan(y_pred) | np.isnan(y_true))
            y_pred, y_true = y_pred[mask], y_true[mask]
            n = len(y_true)
            if n < 5: continue
            hits = int((np.sign(y_pred) == np.sign(y_true)).sum())
            da = hits / n
            p_binom = sstats.binomtest(hits, n, 0.5, alternative="greater").pvalue
            ci_lo, ci_hi = wilson_ci(hits, n)
            # corr is undefined when either side is constant (e.g. the `zero`
            # baseline predicts an all-zero, zero-variance vector) → NaN, and we
            # skip it to avoid the noisy RuntimeWarning / ConstantInputWarning.
            const_input = n < 2 or np.std(y_pred) == 0 or np.std(y_true) == 0
            r_p = np.nan if const_input else float(np.corrcoef(y_pred, y_true)[0,1])
            r_s = np.nan if const_input else float(sstats.spearmanr(y_pred, y_true).correlation)
            amp = float(np.median(np.abs(y_pred) / (np.abs(y_true) + 1e-12)))
            q_lo = slab[lo_q].values[mask]
            q_hi = slab[hi_q].values[mask]
            cov = float(((y_true >= q_lo) & (y_true <= q_hi)).mean())
            mae = float(np.mean(np.abs(y_true - y_pred)))
            rows.append(dict(ticker=tic, horizon=h, n=n, da=da, p_binom=p_binom,
                             da_ci_lo=ci_lo, da_ci_hi=ci_hi, pearson=r_p, spearman=r_s,
                             amp_ratio=amp, coverage=cov, mae=mae))
    out = pd.DataFrame(rows)
    if not out.empty:
        # BH-adjusted q across the (ticker, horizon) cells
        m = len(out)
        order = out["p_binom"].rank(method="first").astype(int).values
        out["p_binom_bh"] = np.minimum.accumulate((out["p_binom"].sort_values().values * m / np.arange(1, m+1))[::-1])[::-1][order-1]
    return out

def aggregate_metrics(per_cell):
    if per_cell.empty: return pd.DataFrame()
    agg = per_cell.groupby("horizon").agg(
        mean_da=("da","mean"), median_da=("da","median"),
        mean_pearson=("pearson","mean"), mean_spearman=("spearman","mean"),
        mean_amp=("amp_ratio","mean"), mean_cov=("coverage","mean"),
        n_cells=("ticker","count"),
        n_signif_05=("p_binom_bh", lambda s: int((s<0.05).sum())),
    ).reset_index()
    return agg


In [ ]:
def trend2_metrics(pred_df, ret_panel, tickers, quantiles):
    """2-bar directional 'trend' agreement on the first two horizon steps.

    A window emits a *trend signal* for a ticker when the predicted median
    return of bar 1 and bar 2 share the same non-zero sign (+,+ or -,-), i.e.
    Chronos calls a 2-bar continuation. Restricted to those signal windows we
    score the realised returns two ways:
      both : real bar1 AND real bar2 each match the predicted sign — the strict
             "+,+ predicted → +,+ realised" reading.  Chance level = 0.25.
      cum  : the 2-bar cumulative real move sign(r1+r2) matches the predicted
             sign — "did the net move go the predicted way".  Chance level = 0.50.
    Also reports the no-signal share (windows where the first two predicted bars
    disagree), since a near-zero median makes signs noisy.
    """
    if pred_df.empty or "id" not in pred_df.columns:
        return pd.DataFrame()
    median_q = str(quantiles[len(quantiles)//2])
    rows = []
    for tic in tickers:
        sub = pred_df[pred_df["id"] == tic].sort_values(["window", "timestamp"])
        if sub.empty: continue
        sub = sub.copy()
        sub["h_step"] = sub.groupby("window").cumcount() + 1
        w1 = sub[sub["h_step"] == 1].set_index("window")
        w2 = sub[sub["h_step"] == 2].set_index("window")
        common = w1.index.intersection(w2.index)
        n_win = len(common)
        n_sig = n_both = n_cum = 0
        for w in common:
            p1, p2 = w1.at[w, median_q], w2.at[w, median_q]
            s = np.sign(p1)
            if s == 0 or np.sign(p2) != s:      # first two predicted bars disagree → no trend signal
                continue
            try:
                t1 = ret_panel[tic].loc[w1.at[w, "timestamp"]]
                t2 = ret_panel[tic].loc[w2.at[w, "timestamp"]]
            except KeyError:
                continue
            if np.isnan(t1) or np.isnan(t2): continue
            n_sig += 1
            if np.sign(t1) == s and np.sign(t2) == s: n_both += 1
            if np.sign(t1 + t2) == s:                 n_cum += 1
        if n_sig == 0: continue
        p_both = sstats.binomtest(n_both, n_sig, 0.25, alternative="greater").pvalue
        p_cum  = sstats.binomtest(n_cum,  n_sig, 0.50, alternative="greater").pvalue
        rows.append(dict(
            ticker=tic, n_windows=n_win, n_trend_signals=n_sig,
            signal_rate=n_sig/n_win if n_win else np.nan,
            n_both=n_both, acc_both=n_both/n_sig, p_both=p_both,
            n_cum=n_cum,   acc_cum=n_cum/n_sig,   p_cum=p_cum,
        ))
    return pd.DataFrame(rows)


## 10. Baselines

Computed on the same walk-forward windows. The runner persists their predictions in the same shape as Chronos's so the metric function reuses the exact same code path.

In [ ]:
def baseline_predictions(name, ret_panel, tickers, anchors, context_len, horizon):
    rows = []
    for i, t in enumerate(anchors):
        pos = ret_panel.index.get_loc(t)
        for tic in tickers:
            ctx = ret_panel[tic].iloc[pos-context_len:pos].values
            fut_ts = ret_panel.index[pos:pos+horizon]
            if name == "zero":
                yhat = np.zeros(horizon)
            elif name == "last":
                yhat = np.full(horizon, ctx[-1] if len(ctx) else 0.0)
            elif name == "momentum5":
                yhat = np.full(horizon, np.nanmean(ctx[-5:]) if len(ctx)>=5 else 0.0)
            elif name == "ar1":
                if len(ctx) >= 30:
                    x, y = ctx[:-1], ctx[1:]
                    a = np.dot(x, y) / (np.dot(x, x) + 1e-12)
                    yhat = np.empty(horizon); last = ctx[-1]
                    for k in range(horizon):
                        last = a * last; yhat[k] = last
                else:
                    yhat = np.zeros(horizon)
            else:
                raise ValueError(name)
            for k, ts in enumerate(fut_ts):
                rows.append(dict(id=tic, timestamp=ts, window=i, t_anchor=t, **{"0.5": yhat[k], "0.1": yhat[k], "0.9": yhat[k]}))
    return pd.DataFrame(rows)


## 11. Plot helpers

In [ ]:
def plot_da_heatmap(per_cell, out_path, title=""):
    pivot = per_cell.pivot(index="ticker", columns="horizon", values="da")
    fig, ax = plt.subplots(figsize=(1.2*len(pivot.columns)+2, 0.4*len(pivot.index)+2))
    im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0.40, vmax=0.60, aspect="auto")
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i,j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, label="Dir. Acc.")
    ax.set_title(title or "Directional accuracy (ticker × horizon)")
    plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

def plot_da_vs_baseline(per_cell, per_cell_baseline, baseline_name, out_path):
    m = per_cell.merge(per_cell_baseline, on=["ticker","horizon"], suffixes=("","_b"))
    m["delta"] = m["da"] - m["da_b"]
    fig, ax = plt.subplots(figsize=(10, 4))
    for h, sub in m.groupby("horizon"):
        ax.bar(sub["ticker"] + f"|h={h}", sub["delta"], label=f"h={h}")
    ax.axhline(0, color="black", lw=0.7)
    ax.set_ylabel(f"DA(Chronos) - DA({baseline_name})")
    ax.tick_params(axis="x", rotation=80, labelsize=7)
    ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

def plot_corr_hist(pred_df, ret_panel, horizons, quantiles, out_path):
    median_q = str(quantiles[len(quantiles)//2])
    fig, axes = plt.subplots(1, len(horizons), figsize=(4*len(horizons), 3), sharey=True)
    if len(horizons) == 1: axes = [axes]
    pred_df = pred_df.sort_values(["window","id","timestamp"])
    pred_df["h_step"] = pred_df.groupby(["window","id"]).cumcount() + 1
    for ax, h in zip(axes, horizons):
        rs = []
        slab = pred_df[pred_df["h_step"]==h]
        for (w, _id), s in slab.groupby(["window","id"]):
            y_p = s[median_q].values
            try: y_t = ret_panel[_id].loc[s["timestamp"].values].values
            except KeyError: continue
            if len(y_t) > 1 and not np.isnan(y_p).any():
                rs.append(np.corrcoef(y_p, y_t)[0,1])
        ax.hist(rs, bins=30); ax.axvline(0, color="red", lw=0.8)
        ax.set_title(f"h={h}, n={len(rs)}"); ax.set_xlabel("Pearson r")
    plt.suptitle("Per-window correlation (pred vs true), by horizon")
    plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

def plot_amplitude(per_cell, out_path):
    fig, ax = plt.subplots(figsize=(8,4))
    for h, sub in per_cell.groupby("horizon"):
        ax.scatter([h]*len(sub), sub["amp_ratio"], label=f"h={h}", alpha=0.6)
    ax.axhline(1.0, color="black", lw=0.6, ls="--")
    ax.set_xlabel("horizon"); ax.set_ylabel("median |pred|/|true|")
    ax.set_yscale("log"); ax.set_title("Amplitude calibration (target = 1.0)")
    plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

def plot_coverage(per_cell, out_path, target=0.80):
    pivot = per_cell.pivot(index="ticker", columns="horizon", values="coverage")
    fig, ax = plt.subplots(figsize=(1.2*len(pivot.columns)+2, 0.4*len(pivot.index)+2))
    im = ax.imshow(pivot.values, cmap="coolwarm", vmin=0.60, vmax=1.0, aspect="auto")
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i,j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, label=f"q-coverage (target={target})")
    ax.set_title("Quantile coverage (q10–q90)")
    plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

def plot_forecast_examples(pred_df, ret_panel, price_panel, tickers, quantiles, out_path,
                           n_examples=6, n_hist=40):
    """Price-space forecast examples on a *positional* x-axis (sequential bar
    index, equal spacing) — NOT real datetime. MOEX intraday bars are 1h apart
    inside the 10:00–17:00 session but separated by 17h overnight / ~65h weekend
    gaps; on a datetime axis matplotlib draws one straight line across each gap,
    so >50% of the width becomes misleading diagonal "ramps" across closed-market
    time. Collapsing to bar index removes that artifact (standard trading-chart
    convention). A few ticks are still date-labelled for orientation.
    The median is cumsum-exp of predicted log-returns, so on intraday data it is
    expected to be ~flat; signal lives in the q10–q90 band and sign."""
    median_q = str(quantiles[len(quantiles)//2])
    lo_q, hi_q = str(quantiles[0]), str(quantiles[-1])
    windows = pred_df["window"].unique()
    pick = np.linspace(0, len(windows)-1, n_examples).astype(int)
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    tic = tickers[0]
    for ax, w in zip(axes.flat, [windows[i] for i in pick]):
        slab = pred_df[pred_df["window"]==w]
        s = slab[slab["id"]==tic].sort_values("timestamp")
        if s.empty: continue
        fut_idx = pd.DatetimeIndex(s["timestamp"].values)
        anchor_pos = price_panel.index.get_loc(fut_idx[0])
        last_p = price_panel[tic].iloc[anchor_pos - 1]
        p_med = last_p * np.exp(np.cumsum(s[median_q].values))
        p_lo  = last_p * np.exp(np.cumsum(s[lo_q].values))
        p_hi  = last_p * np.exp(np.cumsum(s[hi_q].values))
        truth = price_panel[tic].loc[fut_idx]
        hist_idx = price_panel.index[max(0, anchor_pos - n_hist):anchor_pos]
        # positional x-axis: history 0..H-1, forecast H..H+horizon-1
        H = len(hist_idx)
        x_hist = np.arange(H)
        x_fut = np.arange(H, H + len(fut_idx))
        ax.plot(x_hist, price_panel[tic].loc[hist_idx].values, color="#444", lw=1, label="history")
        ax.axvline(H - 0.5, ls="--", color="gray", lw=1)
        ax.plot(x_fut, truth.values, "g.-", label="actual")
        ax.plot(x_fut, p_med, "r.-", label="median")
        ax.fill_between(x_fut, p_lo, p_hi, color="red", alpha=0.15)
        # date-labelled ticks for orientation (bar index axis stays equal-spaced)
        all_idx = hist_idx.append(fut_idx)
        tick_pos = np.linspace(0, len(all_idx) - 1, 5).astype(int)
        ax.set_xticks(tick_pos)
        ax.set_xticklabels([all_idx[p].strftime("%m-%d %H:%M") for p in tick_pos], fontsize=6, rotation=30)
        ax.set_title(f"{tic}  window={w}"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(out_path, dpi=120); plt.close()

## 12. Stage orchestrator

`run_stage(cfg)` is the only thing `run.ipynb` calls per stage. It snapshots the config, prefetches missing data (idempotent), runs walk-forward, computes metrics + baselines, writes plots, dumps a one-line summary.

In [ ]:
def run_stage(cfg_path: str, cache_dir: Optional[str] = None, pipeline=None):
    cfg = load_config(cfg_path)
    out_dir = Path(cfg["output_dir"]) / cfg["stage_id"]
    out_dir.mkdir(parents=True, exist_ok=True)
    save_config_snapshot(cfg, str(out_dir))
    cache_dir = cache_dir or resolve_cache_dir(cfg)
    # 1. prefetch (idempotent)
    manifest = build_prefetch_manifest([cfg])
    prefetch_all(manifest, cache_dir)
    # 2. cache-only load
    prices, indexes, futures = load_stage_inputs(cfg, cache_dir)
    price_panel, ret_panel, cov_panel = assemble_panels(cfg, prices, indexes, futures)
    print(f"panels: price={price_panel.shape} ret={ret_panel.shape} cov={cov_panel.shape}")
    # build_price_panel's min_ticker_coverage guard may have dropped tickers (e.g.
    # short-history recent listings) -- reconcile cfg["tickers"] to the survivors
    # (ret_panel.columns) now, once, so every downstream consumer (walk-forward,
    # baselines, metrics, plots) sees the same ticker list ret_panel actually has.
    # Before this fix: run_walk_forward -> build_chronos_inputs indexed ret_panel[tic]
    # for every ORIGINALLY CONFIGURED ticker, crashing with KeyError on the first
    # dropped one (caught locally 2026-09-17 on the real Phase B panel, which drops
    # 6 of 22 configured tickers).
    dropped = [t for t in cfg["tickers"] if t not in ret_panel.columns]
    if dropped:
        print(f"  run_stage: {len(dropped)} configured tickers not in the final panel "
              f"(dropped upstream, e.g. by min_ticker_coverage): {dropped}")
    cfg["tickers"] = [t for t in cfg["tickers"] if t in ret_panel.columns]
    # 3. pipeline
    if pipeline is None:
        from chronos import Chronos2Pipeline
        pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE, torch_dtype=DTYPE)
    # 4. walk-forward
    preds = run_walk_forward(pipeline, cfg, ret_panel, cov_panel, str(out_dir))
    # 5. baselines
    anchors = walk_forward_anchors(ret_panel.index, cfg["context_len"], cfg["horizon"],
                                   shift=cfg["walk_forward"]["shift"], max_windows=cfg["walk_forward"].get("max_windows"))
    base_preds = {}
    for b in cfg["baselines"]:
        base_preds[b] = baseline_predictions(b, ret_panel, cfg["tickers"], anchors, cfg["context_len"], cfg["horizon"])
    # 6. metrics
    per_cell = per_cell_metrics(preds, ret_panel, cfg["tickers"], cfg["eval_horizons"], cfg["quantiles"])
    per_cell.to_csv(out_dir / "metrics.csv", index=False)
    agg = aggregate_metrics(per_cell); agg.to_csv(out_dir / "metrics_aggregate.csv", index=False)
    tr2 = trend2_metrics(preds, ret_panel, cfg["tickers"], cfg["quantiles"])
    tr2.to_csv(out_dir / "metrics_trend2.csv", index=False)
    base_rows = []
    for b, bp in base_preds.items():
        m = per_cell_metrics(bp, ret_panel, cfg["tickers"], cfg["eval_horizons"], cfg["quantiles"])
        m["baseline"] = b; base_rows.append(m)
    if base_rows:
        pd.concat(base_rows, ignore_index=True).to_csv(out_dir / "metrics_baselines.csv", index=False)
    # 7. plots
    plots = out_dir / "plots"; plots.mkdir(exist_ok=True)
    plot_da_heatmap(per_cell, plots / "da_heatmap.png", title=cfg["stage_id"])
    if "last" in base_preds:
        per_b = per_cell_metrics(base_preds["last"], ret_panel, cfg["tickers"], cfg["eval_horizons"], cfg["quantiles"])
        plot_da_vs_baseline(per_cell, per_b, "last", plots / "da_vs_last.png")
    plot_corr_hist(preds, ret_panel, cfg["eval_horizons"], cfg["quantiles"], plots / "corr_hist.png")
    plot_amplitude(per_cell, plots / "amplitude.png")
    plot_coverage(per_cell, plots / "coverage.png")
    plot_forecast_examples(preds, ret_panel, price_panel, cfg["tickers"], cfg["quantiles"], plots / "examples.png")
    # 8. summary
    primary = per_cell[per_cell["horizon"].isin(cfg["primary_horizons"])]
    summary = dict(
        stage_id=cfg["stage_id"],
        group_mode=cfg.get("group_mode", "univariate"),
        n_windows=int(preds["window"].nunique()) if not preds.empty else 0,
        mean_da_primary=float(primary["da"].mean()) if not primary.empty else None,
        median_da_primary=float(primary["da"].median()) if not primary.empty else None,
        cells_signif_05=int((primary["p_binom_bh"]<0.05).sum()) if not primary.empty else 0,
        mean_pearson_primary=float(primary["pearson"].mean()) if not primary.empty else None,
        mean_coverage_primary=float(primary["coverage"].mean()) if not primary.empty else None,
    )
    if not tr2.empty:
        tot = int(tr2["n_trend_signals"].sum())
        summary["trend2_n_signals"] = tot
        summary["trend2_acc_both"] = float(tr2["n_both"].sum()/tot) if tot else None  # chance 0.25
        summary["trend2_acc_cum"]  = float(tr2["n_cum"].sum()/tot)  if tot else None  # chance 0.50
    with open(out_dir / "summary.json", "w") as f: json.dump(summary, f, indent=2)
    print("SUMMARY:", json.dumps(summary, indent=2))
    return summary

## 13. Basket gate — paired McNemar test

Pre-registered stopping rule for the multivariate-vs-univariate gate (McNemar's test on
paired hit/miss outcomes). Needs two `run_stage` outputs from identical configs except
`group_mode` (a `*_multivariate.yaml` / `*_univariate.yaml` pair, e.g.
`configs/basket_gate_daily_adj_multivariate.yaml`) — same tickers, same windows, same
covariates, so every (ticker, horizon, window) cell is directly paired between the two arms.

For each paired cell: hit = predicted sign matches realised sign (identical rule to
`per_cell_metrics`, replicated here since hit/miss per window isn't itself persisted to
disk — only the aggregated per-cell DA is). McNemar's test looks only at the *discordant*
pairs (multivariate right/univariate wrong, or the reverse) — concordant pairs (both right
or both wrong) carry no information about which arm is better. Run per (ticker, horizon)
cell, BH-corrected across the cell family, matching the BH pattern already used in
`per_cell_metrics`.

In [ ]:
def _hit_miss_by_cell(pred_df, ret_panel, tickers, horizons, quantiles):
    """Per (ticker, horizon, window): 1 if predicted sign matches realised sign, else 0.
    Same sign-match rule as per_cell_metrics, but keyed per window (not aggregated) so
    two arms' outcomes can be paired window-by-window for McNemar's test.
    Returns a dict {(ticker, horizon): {window: hit_bool}}.
    """
    median_q = str(quantiles[len(quantiles)//2])
    out = {}
    if pred_df.empty or "id" not in pred_df.columns:
        return out
    for tic in tickers:
        sub = pred_df[pred_df["id"] == tic].copy()
        if sub.empty:
            continue
        sub = sub.sort_values(["window", "timestamp"])
        sub["h_step"] = sub.groupby("window").cumcount() + 1
        for h in horizons:
            slab = sub[sub["h_step"] == h]
            if slab.empty:
                continue
            y_pred = slab[median_q].values
            ts = slab["timestamp"].values
            windows = slab["window"].values
            try:
                y_true = ret_panel[tic].loc[pd.DatetimeIndex(ts)].values
            except KeyError:
                continue
            mask = ~(np.isnan(y_pred) | np.isnan(y_true))
            hits = (np.sign(y_pred[mask]) == np.sign(y_true[mask]))
            out[(tic, h)] = dict(zip(windows[mask].tolist(), hits.tolist()))
    return out

def mcnemar_gate_test(preds_multi, preds_uni, ret_panel, tickers, horizons, quantiles,
                       alpha=0.05):
    """Paired McNemar test per (ticker, horizon) cell, BH-corrected across the cell
    family -- the Phase B gate's pre-registered stopping rule. `preds_multi`/`preds_uni`
    are the raw `preds.parquet` DataFrames from the two run_stage outputs (identical
    configs except group_mode). Only windows present in BOTH arms for a given cell are
    paired; windows unique to one arm (shouldn't happen if both ran to completion on the
    same config, but guarded) are dropped from that cell's test.

    Returns a per-cell DataFrame (ticker, horizon, n_paired, n01, n10, da_multi, da_uni,
    delta_da, mcnemar_stat, p_value, p_value_bh) plus a `gate` dict with the pre-registered
    pass/fail decision: PASS iff aggregate delta_da > 0 AND >=1 BH-significant cell
    favoring multivariate AND multivariate still beats the `last` baseline on aggregate DA
    (baseline check left to the caller -- pass baseline_da_multi explicitly since it lives
    in metrics_baselines.csv, not preds.parquet).
    """
    hm_multi = _hit_miss_by_cell(preds_multi, ret_panel, tickers, horizons, quantiles)
    hm_uni   = _hit_miss_by_cell(preds_uni,   ret_panel, tickers, horizons, quantiles)
    rows = []
    for cell in sorted(set(hm_multi) & set(hm_uni)):
        tic, h = cell
        wm, wu = hm_multi[cell], hm_uni[cell]
        common_windows = sorted(set(wm) & set(wu))
        if len(common_windows) < 5:
            continue
        hits_m = np.array([wm[w] for w in common_windows])
        hits_u = np.array([wu[w] for w in common_windows])
        n01 = int(((~hits_m) & hits_u).sum())   # multivariate wrong, univariate right
        n10 = int((hits_m & (~hits_u)).sum())   # multivariate right, univariate wrong
        n = len(common_windows)
        # exact binomial McNemar (small-sample safe; matches statsmodels' exact=True default
        # behaviour without adding a new dependency) -- two-sided test on the discordant pairs
        k = min(n01, n10)
        m = n01 + n10
        p_value = 1.0 if m == 0 else min(1.0, 2 * sstats.binom.cdf(k, m, 0.5))
        rows.append(dict(
            ticker=tic, horizon=h, n_paired=n, n01=n01, n10=n10,
            da_multi=float(hits_m.mean()), da_uni=float(hits_u.mean()),
            delta_da=float(hits_m.mean() - hits_u.mean()),
            mcnemar_stat=int(m), p_value=float(p_value),
        ))
    out = pd.DataFrame(rows)
    if out.empty:
        return out, dict(gate="FAIL", reason="no pairable cells (n_paired>=5)")
    # BH correction across the cell family, same pattern as per_cell_metrics
    mtot = len(out)
    order = out["p_value"].rank(method="first").astype(int).values
    out["p_value_bh"] = np.minimum.accumulate(
        (out["p_value"].sort_values().values * mtot / np.arange(1, mtot + 1))[::-1]
    )[::-1][order - 1]
    agg_delta_da = float(out["delta_da"].mean())
    signif_favoring_multi = out[(out["p_value_bh"] < alpha) & (out["delta_da"] > 0)]
    passed = agg_delta_da > 0 and len(signif_favoring_multi) >= 1
    gate = dict(
        gate="PASS" if passed else "FAIL",
        agg_delta_da=agg_delta_da,
        n_cells=mtot,
        n_signif_bh=int((out["p_value_bh"] < alpha).sum()),
        n_signif_favoring_multivariate=len(signif_favoring_multi),
        note="Baseline-beating check (multivariate must still beat the 'last' baseline on "
             "aggregate DA) is NOT evaluated here -- check metrics_baselines.csv separately "
             "per the pre-registered rule before finalizing the gate decision.",
    )
    return out, gate

In [ ]:
def residualize_market_factor(ret_panel):
    """Leave-one-out cross-sectional market residualization. Every ticker's return
    is demeaned by every OTHER ticker's same-bar return (never its own), which
    avoids mechanically inducing negative correlation -- see `pairwise_lagged_xcorr`
    below for the full derivation and the small-N-synthetic-panel caveat this was
    verified against at real-panel scale. Shared by `pairwise_lagged_xcorr` (§14)
    and Phase E's event detector (§16) -- one implementation, not two.
    """
    tickers = list(ret_panel.columns)
    n_tickers = len(tickers)
    if n_tickers <= 2:
        return ret_panel.copy()
    col_sum = ret_panel.sum(axis=1)
    return ret_panel.apply(lambda col: col - (col_sum - col) / (n_tickers - 1))


## 14. Lead-lag screening (discovery + shortlist)

Pure pandas/numpy on log-returns -- no Chronos, no GPU. Two functions:

- `pairwise_lagged_xcorr(ret_panel, max_lag=5)`: for every ticker pair `(i, j)`
  with `i < j` and every lag `1..max_lag`, computes Pearson correlation between
  `ret_panel[i]` and `ret_panel[j]` shifted by the lag, in both directions
  (`j` leads `i`, and `i` leads `j`, tested as separate hypotheses since
  lead-lag is directional). Returns one row per `(ticker_i, ticker_j, lag,
  direction)` with `r`, `p_value` (from `scipy.stats.pearsonr`), `n`.
- `select_pair_shortlist(xcorr_results, alpha=0.05, top_n=20)`: BH-corrects
  `p_value` across the full test family (same 5-line pattern as
  `mcnemar_gate_test` in section 13 -- copied verbatim, not reimplemented),
  filters to `p_value_bh < alpha`, caps at `top_n` by `|r|` if oversubscribed.

Caller is responsible for calling `ret_panel = log_returns(build_price_panel(...))`
on the **discovery-window slice only** (a fixed, pre-registered date range that ends
before the confirmation window begins -- see the `leadlag_confirm_*.yaml` configs'
header comments for the exact split) before passing it in -- these two functions have
no knowledge of the split themselves.

In [ ]:
def pairwise_lagged_xcorr(ret_panel, max_lag=5, residualize_market=True):
    """Pearson cross-correlation for every ticker pair at every lag, both directions.

    For pair (i, j), i<j:
      direction="j_leads_i": corr(ret_panel[i], ret_panel[j].shift(lag))
        -- does j's past return predict i's return `lag` bars later.
      direction="i_leads_j": corr(ret_panel[j], ret_panel[i].shift(lag))
        -- does i's past return predict j's return `lag` bars later.
    Both are tested since lead-lag is directional; a pair can show one, both,
    or neither. NaNs from the shift are dropped pairwise (not panel-wide) so
    n varies slightly by lag but every row's n is exact, not estimated.

    `residualize_market`: if True (default), every series is first demeaned
    by its own leave-one-out cross-sectional average (every OTHER ticker's
    same-day return, i.e. `(sum(all) - self) / (n_tickers - 1)`) before
    lagged correlation is computed. Added 2026-09-17 after an initial run
    without this on the real 80-ticker panel found 20 BH-significant
    pairs, 19/20 sharing the SAME lag (3) across otherwise-unrelated
    tickers -- the panel-wide market-average return itself has lag-3
    autocorrelation ~0.16, which alone explains a broad cluster of
    spurious same-lag hits across many pairs.

    Leave-one-out (not plain `ret_panel.mean(axis=1)`, which includes the
    ticker being residualized) matters: including a ticker in its own
    "market average" mechanically induces negative correlation between it
    and everything else purely from the shared subtraction. Verified this
    is negligible at the real universe's actual scale (~55-80 tickers:
    induced same-day correlation between two independent series ≈ -0.02,
    and no spurious noise-pair survives BH correction across the full test
    family) but NOT negligible on a small (~10-ticker) synthetic sanity
    panel, where it's large enough to produce spurious BH-significant
    hits -- so the synthetic verification test for this function must use
    a ticker count comparable to the real panel (~50+), a small synthetic
    panel is a harder/unrepresentative regime, not a conservative check.

    The residualization itself is `residualize_market_factor` (§16), extracted
    there so this function and Phase E's detector share one implementation.
    """
    tickers = list(ret_panel.columns)
    n_tickers = len(tickers)
    panel = residualize_market_factor(ret_panel) if residualize_market else ret_panel
    rows = []
    for a_idx in range(n_tickers):
        for b_idx in range(a_idx + 1, n_tickers):
            tic_i, tic_j = tickers[a_idx], tickers[b_idx]
            for lag in range(1, max_lag + 1):
                for direction, leader, follower in (
                    ("j_leads_i", tic_j, tic_i),
                    ("i_leads_j", tic_i, tic_j),
                ):
                    x = panel[follower]
                    y = panel[leader].shift(lag)
                    mask = ~(x.isna() | y.isna())
                    n = int(mask.sum())
                    if n < 30:  # need enough pairs for pearsonr's t-distribution to be meaningful
                        continue
                    r, p_value = sstats.pearsonr(x[mask].values, y[mask].values)
                    rows.append(dict(
                        ticker_i=tic_i, ticker_j=tic_j, lag=lag, direction=direction,
                        leader=leader, follower=follower,
                        r=float(r), p_value=float(p_value), n=n,
                    ))
    return pd.DataFrame(rows)


def select_pair_shortlist(xcorr_results, alpha=0.05, top_n=20):
    """BH-correct across the full (pair, lag, direction) test family, shortlist survivors.

    BH pattern copied verbatim from `mcnemar_gate_test` (section 13) -- same
    5-line implementation, only column names changed, to avoid a third
    slightly-different BH implementation in this codebase.
    """
    if xcorr_results.empty:
        return xcorr_results.assign(p_value_bh=[]), pd.DataFrame()
    out = xcorr_results.copy()
    mtot = len(out)
    order = out["p_value"].rank(method="first").astype(int).values
    out["p_value_bh"] = np.minimum.accumulate(
        (out["p_value"].sort_values().values * mtot / np.arange(1, mtot + 1))[::-1]
    )[::-1][order - 1]
    signif = out[out["p_value_bh"] < alpha].copy()
    if len(signif) > top_n:
        signif = signif.reindex(signif["r"].abs().sort_values(ascending=False).index).head(top_n)
    shortlist = signif.sort_values("p_value_bh").reset_index(drop=True)
    return out, shortlist

## 15. Quantile (pinball) loss

Added 2026-09-17 after three DA/Pearson-based negative results (Stage 2b,
Phase B, Phase C) raised the question of whether DA's binary sign threshold
is masking real calibration/quantile skill -- Chronos-2 outputs full quantile
forecasts, and DA only ever looks at the median. `pinball_loss`/
`per_cell_quantile_loss` compute the standard mean pinball loss across all
configured quantiles from a run's saved `preds.parquet` (median + q10/q90 by
default), independent of `per_cell_metrics` so it can be applied
retroactively to already-concluded runs (Phase B, Phase C) without
recomputing anything else.

No baseline-comparison helper: `baseline_predictions` (section 10) gives
every baseline the same value across all three quantile columns (they're
point forecasts, not distributional ones), so a baseline's own "pinball
loss" would just be a rescaled MAE, not a real calibration comparison.
Interpret pinball loss jointly with `per_cell_metrics`'s `coverage` column
instead (q10-q90 hit rate vs the target 0.80).

In [ ]:
def pinball_loss(y_true, quantile_preds, quantiles):
    """Mean pinball (quantile) loss across all configured quantiles.

    `quantile_preds` is a dict {quantile: array of predicted values}, same
    shape as `y_true`. Standard definition: for quantile tau,
    L_tau(y, yhat) = max(tau*(y-yhat), (tau-1)*(y-yhat)); this returns the
    mean over both quantiles and samples. Lower is better.

    No baseline-comparison helper is provided here: `baseline_predictions`
    (section 10) sets all three quantile columns to the same point value for
    every baseline (zero/last/momentum5/ar1 are point forecasts, not
    distributional ones), so a baseline's "pinball loss" would just be a
    rescaled version of its own MAE (already in `metrics.csv`), not a real
    calibration comparison -- pinball loss is only informative measured
    against `per_cell_metrics`'s `coverage` column (q10-q90 hit rate vs the
    target 0.80) as a joint read: low pinball loss + coverage far from 0.80
    means overconfident/underconfident intervals, not skill.
    """
    losses = []
    for q in quantiles:
        yhat = quantile_preds[q]
        diff = y_true - yhat
        losses.append(np.maximum(q * diff, (q - 1) * diff))
    return float(np.mean(np.stack(losses)))


def per_cell_quantile_loss(pred_df, ret_panel, tickers, horizons, quantiles):
    """Per (ticker, horizon) mean pinball loss -- companion to `per_cell_metrics`,
    kept separate rather than folded in since it's a 2026-09-17 addition (see
    docs/current_state.md) applied both retroactively (existing runs' saved
    preds.parquet) and going forward, and callers may want it without
    recomputing the rest of per_cell_metrics.
    """
    if pred_df.empty or "id" not in pred_df.columns:
        return pd.DataFrame()
    rows = []
    for tic in tickers:
        sub = pred_df[pred_df["id"] == tic].copy()
        if sub.empty: continue
        sub = sub.sort_values(["window", "timestamp"])
        sub["h_step"] = sub.groupby("window").cumcount() + 1
        for h in horizons:
            slab = sub[sub["h_step"] == h]
            if slab.empty: continue
            ts = slab["timestamp"].values
            try:
                y_true = ret_panel[tic].loc[pd.DatetimeIndex(ts)].values
            except KeyError:
                continue
            q_cols = [str(q) for q in quantiles]
            if not all(c in slab.columns for c in q_cols):
                continue
            preds_by_q = {q: slab[str(q)].values for q in quantiles}
            mask = ~np.isnan(y_true)
            for q in quantiles:
                mask &= ~np.isnan(preds_by_q[q])
            n = int(mask.sum())
            if n < 5: continue
            y_true_m = y_true[mask]
            preds_m = {q: preds_by_q[q][mask] for q in quantiles}
            loss = pinball_loss(y_true_m, preds_m, quantiles)
            rows.append(dict(ticker=tic, horizon=h, n=n, pinball_loss=loss))
    return pd.DataFrame(rows)

## 16. Event-conditioned burst detection (synthetic validation + real-data MVP)

Detection-only follow-on after four full-sample negative results (an early pairwise
multivariate-basket smoke test, the daily/1h basket gate, and the daily/1h lead-lag
confirmation screen — all gate FAILED). Tests a
different hypothesis: dependencies may not be stationary across the whole sample but occur
in short, real bursts that a full-sample correlation/DA number averages away. Design and
scope discussed with the user 2026-09-17; full background, deferred experiment families
(dynamic networks, change-point-first, nonlinear/Hawkes methods, natural experiments — not
built here), and the statistical-safeguards checklist this section follows are in
`transient_dependency_research.md`.

**Scope locked in for this pass**: synthetic positive-control validation, then a single
event-conditioned leader/follower MVP on the already-pulled 76-ticker 1h panel.
Deferred: 10-minute-or-finer resolution, the broader mechanism-specific covariate list
(order flow, news flags, futures basis — not in `data_pipeline` yet), and every experiment
family beyond A (event-conditioned) in the research doc.

**Event-count floor (derived, not guessed)**: targeting a 65% same-direction follower
response rate vs. the 50% no-relationship null (a 15-point edge — the smallest effect
size treated as economically/scientifically interesting for this pass) requires n≥85
qualifying leader-events per candidate pair for 80% power at α=0.05 (binomial two-proportion
power calculation). Checked against real data: a one-sided 2σ leader-event threshold on
SBER over the 1h discovery window (~2900 bars) yields ~72 expected events — close to but
under the floor, meaning this first pass will realistically only be well-powered for a
handful of the most liquid tickers as leaders. Pairs below the floor are marked ineligible,
not tested — this is a real scope constraint of using 1h bars with a ~1-year discovery
window, not an oversight.

`residualize_market_factor` is defined once, just before §14 (its first use point), and
shared by `pairwise_lagged_xcorr` (§14) and every function below — one implementation,
not duplicated, same verified behavior at real-panel scale.


In [ ]:
def detect_leader_events(residual_series, threshold_std=2.0, direction="both", min_history=60):
    """Causal (trailing-only) leader-event flags for one residualized return series.

    At each bar t, computes the rolling std of the PAST `min_history` bars
    (window ending at t-1, never including t or later -- real-time legality,
    per transient_dependency_research.md safeguard #2) and flags an event
    if the bar-t return exceeds `threshold_std` trailing standard deviations.
    The first `min_history` bars can never be flagged (no valid trailing window
    yet) -- returned as False, not NaN, so callers can treat the output as a
    plain boolean mask without special-casing the warm-up period.

    `direction`: "both" flags |return| > threshold*std (either sign, tested as
    one event family); "positive"/"negative" flags only that sign -- doc's
    Experiment A step 2 treats these as separate hypotheses when both are used,
    so a caller wanting both signs separately should call this twice, not rely
    on "both" internally splitting them.
    """
    assert direction in ("both", "positive", "negative")
    trailing_std = residual_series.shift(1).rolling(min_history, min_periods=min_history).std()
    z = residual_series / trailing_std
    if direction == "both":
        flag = z.abs() > threshold_std
    elif direction == "positive":
        flag = z > threshold_std
    else:
        flag = z < -threshold_std
    return flag.fillna(False)


def event_conditioned_response(residual_panel, leader, follower, lag, threshold_std=2.0,
                                direction="both", min_history=60, min_events=85):
    """Core E2 detector statistic for one (leader, follower, lag) hypothesis.

    Finds every bar where `leader`'s residualized return clears a trailing-std
    threshold (a "leader event", real-time legal per `detect_leader_events`),
    then looks at `follower`'s residualized return `lag` bars later. Returns a
    dict with the qualifying event count, the fraction showing a same-direction
    response (leader event sign == follower response sign), a two-sided
    binomial p-value against the 50% no-relationship null, and `eligible`
    (n_events >= min_events -- see §16 header for how min_events=85 was
    derived). Ineligible pairs are still returned (with eligible=False) so
    callers can report "not enough events" distinctly from "tested and failed."
    """
    events = detect_leader_events(residual_panel[leader], threshold_std, direction, min_history)
    event_idx = residual_panel.index[events]
    # Follower response `lag` bars after each event, by integer position (not timedelta
    # arithmetic) -- robust to the session-grid gaps (overnight/weekend) baked into the
    # 1h panel's DatetimeIndex, where "index[p] + lag hours" would land on a non-trading
    # timestamp that isn't in the index at all.
    pos = {ts: i for i, ts in enumerate(residual_panel.index)}
    event_pos = [pos[ts] for ts in event_idx]
    n = len(residual_panel)
    leader_vals, follower_vals = [], []
    for p in event_pos:
        fp = p + lag
        if fp >= n:
            continue
        leader_vals.append(residual_panel[leader].iloc[p])
        follower_vals.append(residual_panel[follower].iloc[fp])
    n_events = len(leader_vals)
    result = dict(leader=leader, follower=follower, lag=lag, direction=direction,
                  threshold_std=threshold_std, n_events=n_events,
                  eligible=n_events >= min_events)
    if n_events == 0:
        result.update(same_dir_frac=np.nan, p_value=np.nan, mean_response=np.nan)
        return result
    leader_arr = np.array(leader_vals)
    follower_arr = np.array(follower_vals)
    same_dir = (np.sign(leader_arr) == np.sign(follower_arr))
    k = int(same_dir.sum())
    same_dir_frac = k / n_events
    p_value = sstats.binomtest(k, n_events, p=0.5, alternative="two-sided").pvalue
    result.update(same_dir_frac=float(same_dir_frac), p_value=float(p_value),
                  mean_response=float(follower_arr.mean()))
    return result
def block_permute_panel(panel, block_size=24, rng=None):
    """Block-shuffle a return panel to build a null distribution, preserving
    within-block serial dependence and cross-sectional structure (a common
    shock still hits every ticker in a shuffled block simultaneously) --
    per transient_dependency_research.md safeguard #5. Shuffles WHOLE
    ROWS (all tickers together) in contiguous blocks of `block_size` bars,
    so a real event that involves a shared market move is preserved as a
    unit; only its position in calendar time is randomized. Trailing bars
    that don't fill a complete block are dropped, not padded.
    """
    rng = np.random.default_rng() if rng is None else rng
    n = len(panel)
    n_blocks = n // block_size
    if n_blocks < 2:
        raise ValueError(f"panel too short ({n} bars) for block_size={block_size}")
    block_order = rng.permutation(n_blocks)
    chunks = [panel.iloc[b*block_size:(b+1)*block_size] for b in block_order]
    out = pd.concat(chunks, ignore_index=False)
    out.index = panel.index[:len(out)]  # relabel with the original (contiguous) index
    return out


def scan_pair_family(residual_panel, candidate_pairs, lags=(1, 2, 3, 4), threshold_std=2.0,
                      direction="both", min_history=60, min_events=85):
    """Runs `event_conditioned_response` for every (leader, follower, lag) combination
    in `candidate_pairs` (a list of (leader, follower) ticker-name tuples -- callers
    decide the candidate list, e.g. all pairs or a liquidity-filtered subset; this
    function doesn't choose candidates itself). Returns one row per (pair, lag).
    Ineligible cells (n_events < min_events) are included with eligible=False, not
    dropped, so the caller can report the floor's practical impact (see §16 header).
    """
    rows = []
    for leader, follower in candidate_pairs:
        for lag in lags:
            rows.append(event_conditioned_response(
                residual_panel, leader, follower, lag, threshold_std, direction,
                min_history, min_events,
            ))
    return pd.DataFrame(rows)


def bh_correct(df, p_col="p_value", alpha=0.05, out_col="p_value_bh"):
    """BH correction, copied verbatim from the pattern in §13/§14 (same 5-line
    implementation everywhere in this codebase -- avoids a fourth slightly-different
    version). Only rows with a non-null p-value participate in the correction;
    ineligible rows (p_value=NaN) get p_value_bh=NaN and are never significant.
    """
    out = df.copy()
    testable = out[p_col].notna()
    mtot = int(testable.sum())
    out[out_col] = np.nan
    if mtot == 0:
        return out
    sub = out.loc[testable, p_col]
    order = sub.rank(method="first").astype(int).values
    bh = np.minimum.accumulate(
        (sub.sort_values().values * mtot / np.arange(1, mtot + 1))[::-1]
    )[::-1][order - 1]
    out.loc[testable, out_col] = bh
    out["significant"] = out[out_col] < alpha
    return out


In [ ]:
def make_synthetic_panel(n_bars=2900, n_tickers=76, market_vol=0.01, idio_vol=0.015,
                          seed=0):
    """Synthetic return panel with realistic common-market structure: every ticker's
    return is `beta * market_factor + idiosyncratic_noise`, beta drawn per ticker from
    a plausible range (0.5-1.5) so the panel isn't a degenerate single-factor model.
    Scale (76 tickers, ~2900 bars) matches the real 1h discovery panel on purpose --
    residualization behavior differs materially at small N (see §14's docstring), so a
    representative test needs a representative scale, not a toy 10-ticker panel.
    No injected pairwise structure by default (`inject_burst` adds it on top).
    """
    rng = np.random.default_rng(seed)
    tickers = [f"SYN{i:03d}" for i in range(n_tickers)]
    idx = pd.RangeIndex(n_bars)  # position-based; event_conditioned_response uses
                                 # integer position, not calendar time, so a plain
                                 # RangeIndex is a faithful stand-in for the real
                                 # session-grid DatetimeIndex here.
    market = rng.normal(0, market_vol, n_bars)
    betas = rng.uniform(0.5, 1.5, n_tickers)
    idio = rng.normal(0, idio_vol, (n_bars, n_tickers))
    data = idio + np.outer(market, betas)
    return pd.DataFrame(data, index=idx, columns=tickers)


def inject_burst(panel, leader, follower, lag, start, length, same_dir_frac=0.75,
                  effect_scale=1.5, threshold_std=2.0, min_history=60, seed=0):
    """Overwrite `follower`'s returns in [start, start+length) so that, `lag` bars
    after each of `leader`'s large moves within that window, `follower` responds
    same-direction with probability `same_dir_frac` (a KNOWN, exactly-controlled
    ground truth -- this is what E1 checks the detector can recover).

    Uses the SAME causal event definition `event_conditioned_response` will later
    apply -- events are detected on `leader`'s RESIDUALIZED series (leave-one-out
    market factor removed via `residualize_market_factor` on the full, unmodified
    panel), not the raw series, and with the same `threshold_std`/`min_history` the
    detector uses by default. An earlier version detected events on the raw series
    and separately on a coarse non-causal in-window rule; both disagreed with what
    `event_conditioned_response` actually scans (residualized events), diluting the
    injected signal (as little as ~50% overlap) and producing a spuriously low
    measured detection rate. Matching the definitions exactly means every event the
    detector later finds inside the injection window is one this function modified.

    Only modifies `follower`, not `leader` -- the injected relationship is
    one-directional by construction, matching the E2 hypothesis shape.
    """
    rng = np.random.default_rng(seed)
    out = panel.copy()
    residual_leader = residualize_market_factor(panel)[leader]
    events_mask = detect_leader_events(residual_leader, threshold_std, "both", min_history)
    window_mask = events_mask & (out.index >= start) & (out.index < start + length)
    event_positions = out.index[window_mask]
    for p in event_positions:
        fp_iloc = out.index.get_loc(p) + lag
        if fp_iloc >= len(out):
            continue
        sign = np.sign(residual_leader.loc[p]) if residual_leader.loc[p] != 0 else 1.0
        follow_sign = sign if rng.random() < same_dir_frac else -sign
        magnitude = abs(out[follower].iloc[fp_iloc]) + effect_scale * out[follower].std()
        out.iloc[fp_iloc, out.columns.get_loc(follower)] = follow_sign * magnitude
    return out, event_positions


def e1_power_test(n_trials=20, n_bars=2900, n_tickers=76, lag=2, burst_length=None,
                   same_dir_frac=0.75, threshold_std=2.0, min_events=85, seed=0):
    """E1a: inject a known relationship, check the detector (a) finds it eligible
    (enough events) and (b) reports same_dir_frac and p-value consistent with the
    injected ground truth, across `n_trials` independent synthetic panels. Reports
    detection rate (fraction of trials where the injected pair comes back eligible
    AND BH-significant) and mean estimation error on same_dir_frac.

    `burst_length=None` (default) injects across the FULL panel, not a short
    sub-window -- E1's job is proving the detector mechanism can recover a known
    relationship at all (event-count floor, residualization consistency, p-value
    calibration), which needs enough events to clear the n>=85 floor reliably. A
    short, genuinely burst-like injection (relationship present in only part of the
    panel) is E2's concern once the basic mechanism is trusted, not a stronger E1
    test -- a short synthetic burst mostly tests "is burst_length long enough for
    85 events," not "does the detector work."
    """
    if burst_length is None:
        burst_length = n_bars
    detected, eligible_count, frac_errors = 0, 0, []
    for trial in range(n_trials):
        panel = make_synthetic_panel(n_bars, n_tickers, seed=seed + trial)
        leader, follower = panel.columns[0], panel.columns[1]
        injected, event_positions = inject_burst(
            panel, leader, follower, lag, 0, burst_length,
            same_dir_frac=same_dir_frac, seed=seed + trial,
        )
        residual = residualize_market_factor(injected)
        result = event_conditioned_response(
            residual, leader, follower, lag, threshold_std=threshold_std,
            min_events=min_events,
        )
        if result["eligible"]:
            eligible_count += 1
            if not np.isnan(result["p_value"]) and result["p_value"] < 0.05:
                detected += 1
            frac_errors.append(abs(result["same_dir_frac"] - same_dir_frac))
    return dict(
        n_trials=n_trials, eligible_rate=eligible_count / n_trials,
        detection_rate=detected / n_trials,
        mean_abs_frac_error=float(np.mean(frac_errors)) if frac_errors else np.nan,
    )


def e1_null_false_alert_rate(n_trials=20, n_bars=2900, n_tickers=20, lag=2,
                              threshold_std=2.0, min_events=30, alpha=0.05, seed=0):
    """E1b: on PURE-NULL synthetic panels (no injected structure at all), check the
    false-alert rate across many candidate pairs is close to `alpha` after BH
    correction, and well below alpha per-cell if uncorrected multiplicity would
    otherwise inflate it. Uses a smaller n_tickers (20, not 76) deliberately here --
    this test's cost scales with n_tickers^2 candidate pairs x n_trials, and its
    purpose (checking the correction controls the false-positive rate) does not
    depend on matching the real panel's exact scale the way residualization
    validation did; still comfortably above the "not a tiny toy panel" floor.
    """
    n_pairs = n_tickers * (n_tickers - 1)  # both directions
    total_false_positives_bh, total_tests = 0, 0
    for trial in range(n_trials):
        panel = make_synthetic_panel(n_bars, n_tickers, seed=1000 + seed + trial)
        residual = residualize_market_factor(panel)
        candidates = [(a, b) for a in panel.columns for b in panel.columns if a != b]
        scan = scan_pair_family(residual, candidates, lags=(lag,),
                                 threshold_std=threshold_std, min_events=min_events)
        scan = bh_correct(scan)
        total_false_positives_bh += int(scan["significant"].sum())
        total_tests += int(scan["p_value"].notna().sum())
    return dict(
        n_trials=n_trials, n_pairs_per_trial=n_pairs,
        mean_bh_significant_per_trial=total_false_positives_bh / n_trials,
        total_tests=total_tests,
        overall_false_positive_rate=total_false_positives_bh / total_tests if total_tests else np.nan,
    )


### E1 — synthetic positive-control validation

Before touching real data: (a) prove the detector recovers a known injected burst
(power/delay), and (b) prove it controls the false-alert rate on pure-null synthetic
panels (safeguards #6 and #11 in the research doc). Both use a synthetic panel with
realistic common-market structure (every ticker loads on a shared market factor plus
idiosyncratic noise), matching the real panel's scale (76 tickers) per the same
"small-N synthetic tests are an unrepresentative regime" lesson learned residualizing
Phase C's discovery screen (§14).


In [ ]:
# Run E1 now (pure pandas/numpy, seconds not minutes -- no Chronos, no real data).
# This cell's output is the actual verification evidence for "the detector works
# before we point it at real data" -- re-run it whenever detect_leader_events /
# event_conditioned_response / residualize_market_factor change.

e1_power = e1_power_test()
print("=== E1a: injected-burst power test (20 trials, same_dir_frac=0.75 ground truth) ===")
print(e1_power)
assert e1_power["eligible_rate"] > 0.8, \
    "injected burst not consistently reaching the n>=85 event floor -- check burst_length/threshold_std"
assert e1_power["detection_rate"] > 0.8, \
    "detector isn't recovering a known 75%-same-direction burst often enough to trust on real data"
assert e1_power["mean_abs_frac_error"] < 0.10, \
    "same_dir_frac estimate is too noisy relative to the injected ground truth"

e1_null = e1_null_false_alert_rate()
print("\n=== E1b: false-alert rate on pure-null synthetic panels (20 trials x 380 pairs) ===")
print(e1_null)
assert e1_null["overall_false_positive_rate"] < 0.05, \
    "BH-corrected false-positive rate exceeds the nominal 5% on pure-null data -- do not proceed to E2"

print("\nE1 PASSED: detector recovers injected bursts and controls the false-alert rate.")


### E2 — event-conditioned MVP on real 1h data

Only reached after E1 passes (the assertions above must succeed). Runs
`scan_pair_family` on the real, already-pulled 76-ticker 1h panel
(`data_pipeline/data/processed/candles_1h/shares.parquet`), using the SAME
discovery/confirmation date split already established for Phase C's 1h follow-on
(see the `leadlag_confirm_1h_*.yaml` configs' header comments) — discovery 2023-01-02..2024-05-24,
confirmation 2024-05-27..2024-12-30 — so this is a genuinely fresh test on the
confirmation dates, not reusing anything already looked at for this specific
hypothesis. Candidate pairs are NOT restricted to Phase C's 13-pair lead-lag
shortlist (that was a full-sample linear-correlation screen — a different
mechanism; reusing its shortlist here would bias toward pairs already tested and
failed for a different hypothesis). Instead, candidates are every ordered pair
among tickers whose discovery-window event count clears the n≥85 floor per
`event_conditioned_response` — pairs below the floor are recorded as ineligible,
not silently dropped, so the floor's coverage is visible in the output.

Discovery-phase family is BH-corrected (`bh_correct`); only pairs passing BH at
q<0.05 in discovery proceed to a fresh, independently-computed test on the
confirmation window (own BH correction within that smaller family, not reusing
discovery p-values) — the same discovery/confirmation discipline used throughout
this project.


In [ ]:
def run_e2_discovery(ret_panel, lags=(1, 2, 3, 4), threshold_std=2.0, min_events=85,
                      alpha=0.05):
    """E2 discovery driver: residualizes `ret_panel`, builds the candidate pair list
    (every ordered pair, both directions), scans the full family via
    `scan_pair_family`, BH-corrects, and reports floor coverage alongside the
    shortlist. Returns (scan_df, shortlist_df, coverage_summary) -- coverage_summary
    makes the event-count floor's real impact visible (see §16 header) rather than
    letting ineligible pairs silently vanish from the output.
    """
    residual = residualize_market_factor(ret_panel)
    tickers = list(ret_panel.columns)
    candidates = [(a, b) for a in tickers for b in tickers if a != b]
    scan = scan_pair_family(residual, candidates, lags=lags, threshold_std=threshold_std,
                             min_events=min_events)
    n_total = len(scan)
    n_eligible = int(scan["eligible"].sum())
    scan_bh = bh_correct(scan, alpha=alpha)
    shortlist = scan_bh[scan_bh["significant"]].sort_values("p_value_bh").reset_index(drop=True)
    coverage = dict(
        n_candidate_tests=n_total, n_eligible=n_eligible,
        eligible_frac=n_eligible / n_total if n_total else 0.0,
        n_tickers=len(tickers), n_shortlisted=len(shortlist),
    )
    return scan_bh, shortlist, coverage


In [ ]:
def run_e2_confirmation(ret_panel, shortlist, lags=None, threshold_std=2.0,
                         min_events=85, alpha=0.05):
    """E2 confirmation driver: re-tests exactly the (leader, follower, lag) hypotheses
    in `shortlist` (discovery output) against a DIFFERENT ret_panel slice (the
    confirmation window), with its own independent BH correction within this smaller
    family -- never reuses discovery p-values, matching the Phase C confirmation
    methodology (`docs/current_state.md` entries 19-20/23). A pair below the event
    floor in the confirmation window (fewer events than discovery, e.g. a thinner
    ticker or shorter window) is marked ineligible/FAIL, not dropped, so a null
    confirmation is distinguishable from "we couldn't test it."
    """
    if shortlist.empty:
        return shortlist.assign(p_value_bh_confirm=[], confirmed=[])
    residual = residualize_market_factor(ret_panel)
    rows = []
    for _, hyp in shortlist.iterrows():
        rows.append(event_conditioned_response(
            residual, hyp["leader"], hyp["follower"], int(hyp["lag"]),
            threshold_std=threshold_std, direction=hyp["direction"],
            min_events=min_events,
        ))
    confirm = pd.DataFrame(rows)
    confirm = bh_correct(confirm, alpha=alpha, out_col="p_value_bh_confirm")
    confirm = confirm.rename(columns={"significant": "confirmed"})
    # A pair below the event floor in the confirmation window must FAIL regardless of
    # its p-value -- bh_correct alone doesn't know about `eligible`, so this has to be
    # enforced explicitly, not left to naturally fall out of the BH step. Caught during
    # the daily E2 run (2026-09-17): all 3 discovery candidates had fewer confirmation-
    # window events (44-60) than the pre-registered floor (79), so none should ever
    # have been markable "confirmed" even if one had cleared BH by chance.
    confirm["confirmed"] = confirm["confirmed"].fillna(False) & confirm["eligible"]
    return confirm


## Usage in `run.ipynb`

```python
# Cell 1: import everything above (re-paste sections 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)
# Cell 2:
CONFIG_PATH = "configs/basket_gate_daily_adj_multivariate.yaml"  # example; see configs/ for the full list
summary = run_stage(CONFIG_PATH)
```

To prefetch all configs at once (do this first on a fresh Drive mount):

```python
all_cfgs = [load_config(p) for p in sorted(Path("configs").glob("*.yaml"))]
cache_dir = resolve_cache_dir(all_cfgs[0])
prefetch_all(build_prefetch_manifest(all_cfgs), cache_dir)
```